# Chạy Ollama trên Google Colab qua Cloudflare (KHÔNG CẦN TÀI KHOẢN)
Notebook này giúp bạn đưa phần nặng nhất của AI Chatbot lên GPU miễn phí của Google Colab mà không cần đăng ký.
**HƯỚNG DẪN:** Bạn chỉ cần bấm nút **PLAY (Chạy ô này)** ở ngay bên trái ô code dưới đây và đợi máy xử lý xong tất cả.

In [ ]:
import os
import time
import subprocess
import threading
import re

print("1. Đang tải LÕI phần mềm Ollama (Bản Portable)...")
os.system("curl -L https://ollama.com/download/ollama-linux-amd64 -o ollama")
os.system("chmod +x ollama")

print("2. Đang khởi chạy máy chủ Ollama dưới nền...")
os.system("OLLAMA_HOST=127.0.0.1 ./ollama serve > ollama.log 2>&1 &")
time.sleep(5)

print("3. Đang tải đường hầm Cloudflare...")
os.system("wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
os.system("chmod +x cloudflared-linux-amd64")

def run_cloudflared():
    os.system("./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:11434 > cloudflare.log 2>&1")

threading.Thread(target=run_cloudflared, daemon=True).start()

print("4. Đang tạo link Public (Chờ 8 giây)...")
time.sleep(8)

url = None
try:
    with open('cloudflare.log', 'r') as f:
        content = f.read()
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', content)
        if match:
            url = match.group(1)
except Exception as e:
    pass

print("\n==================================================")
if url:
    print(f"🔥 LINK KẾT NỐI (Hãy copy): {url}")
    print("\n👉 Dán vào file .env ở máy tính của bạn: OLLAMA_HOST=" + url)
else:
    print("Đang tạo link, bạn hãy đợi vài giây rồi mở file 'cloudflare.log' ở cột bên trái của Colab để tự copy link nhé.")
print("==================================================\n")

print("5. Bắt đầu tải mô hình AI...")
print("-> Đang kéo nomic-embed-text...")
os.system("./ollama pull nomic-embed-text")
print("-> Đang kéo qwen2.5:7b...")
os.system("./ollama pull qwen2.5:7b")
print("\n✅ HOÀN TẤT TẤT CẢ! Hệ thống đang chạy...")

print("\n⚠️ LƯU Ý: Vui lòng KHÔNG tắt Tab này và ĐỂ NGUYÊN cho ô code này tiếp tục chạy (Vòng lặp vô hạn) để giữ cho Cloudflare và Ollama không bị Colab tự động tắt.")
while True:
    time.sleep(60)